# 마이크로 학습
- Epoch 1 | 최종 Train Acc: 94.70%
- Epoch 2 | 최종 Train Acc: 99.55%
- Epoch 3 | 최종 Train Acc: 99.65%

In [46]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from pathlib import Path
from PIL import Image
import timm

# 1. 데이터셋 클래스 정의
class BeeDataset(Dataset):
    def __init__(self, image_dir, label_dir, transform=None):
        self.image_paths = []
        self.labels = []
        self.transform = transform
        self.class_to_idx = {"알": 0, "백묵병": 1, "유충_응애": 2, "유충_정상": 3}
        
        for class_name, idx in self.class_to_idx.items():
            img_folder = image_dir / class_name
            lbl_folder = label_dir / class_name
            if img_folder.exists() and lbl_folder.exists():
                json_set = {p.stem for p in lbl_folder.glob("*.json")}
                for img_path in img_folder.glob("*"):
                    if img_path.suffix.lower() in ['.jpg', '.jpeg', '.png', '.bmp']:
                        if img_path.stem in json_set:
                            self.image_paths.append(img_path)
                            self.labels.append(idx)
        print(f"✅ 데이터 매칭 완료: 총 {len(self.image_paths)} 개")

    def __len__(self): return len(self.image_paths)
    def __getitem__(self, idx):
        try:
            img = Image.open(self.image_paths[idx]).convert('RGB')
            label = self.labels[idx]
            if self.transform: img = self.transform(img)
            return img, label
        except: return self.__getitem__((idx + 1) % len(self.image_paths))

# 2. 모델 및 트랜스폼 설정
model = timm.create_model('hf_hub:timm/efficientnet_b0.ra_in1k', pretrained=True, num_classes=4)
transform = timm.data.create_transform(**timm.data.resolve_model_data_config(model))

# 3. 데이터셋 및 로더 생성 (경로가 정확한지 다시 한번 확인!)
DATA_ROOT = Path(r"C:\Project\PyCharmMiscProject\PythonProject02\organized_bee_dataset")
dataset = BeeDataset(DATA_ROOT / "train", DATA_ROOT / "labels", transform=transform)
val_dataset = BeeDataset(DATA_ROOT / "val_train", DATA_ROOT / "val_labels", transform=transform)

train_loader = DataLoader(dataset, batch_size=32, shuffle=True, num_workers=0, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=0, pin_memory=True)

print("🚀 데이터셋과 로더가 성공적으로 준비되었습니다! 이제 학습 루프를 실행하세요.")

✅ 데이터 매칭 완료: 총 110974 개
✅ 데이터 매칭 완료: 총 13145 개
🚀 데이터셋과 로더가 성공적으로 준비되었습니다! 이제 학습 루프를 실행하세요.


In [47]:
model = timm.create_model('microsoft/resnet-50', pretrained=True, num_classes=4)
model = model.to(DEVICE)
transform = timm.data.create_transform(**timm.data.resolve_model_data_config(model))

# 4. 데이터셋 및 데이터로더 생성
dataset = BeeDataset(IMAGE_DIR, LABEL_DIR, transform=transform)

# ⚠️ 주피터 무한 대기 버그를 피하기 위해 num_workers는 0으로 하되, 
# 배치 사이즈(128)와 GPU 가속(pin_memory, GradScaler)으로 속도를 커버합니다.
loader = DataLoader(
    dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=True, 
    num_workers=0,       
    pin_memory=True
)

# 5. 최적화 도구 설정
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
criterion = nn.CrossEntropyLoss()
scaler = torch.amp.GradScaler('cuda')

# 6. 학습 루프 (스크립트 형태 그대로 순차 실행)
for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    correct = 0
    
    pbar = tqdm(loader, desc=f"Epoch {epoch + 1}/{EPOCHS}")
    for images, labels in pbar:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        
        optimizer.zero_grad()
        
        # 혼합 정밀도 가속 (학습 속도를 대폭 끌어올립니다)
        with torch.amp.autocast('cuda'):
            outputs = model(images)
            loss = criterion(outputs, labels)
            
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == labels).sum().item()
        running_loss += loss.item()
        
        # 실시간 수치 표기
        pbar.set_postfix(
            loss=f"{loss.item():.4f}", 
            acc=f"{(correct / ((pbar.n + 1) * BATCH_SIZE)) * 100:.2f}%"
        )
        
    scheduler.step()
    print(f"▶ Epoch {epoch + 1} 종료 | Loss: {running_loss / len(loader):.4f} | Acc: {(correct / len(dataset)) * 100:.2f}%")

# 7. 최종 모델 저장
torch.save(model.state_dict(), 'microsoft/resnet-50_bee_model.pth')
print("💾 학습 완료! 최적화된 모델이 저장되었습니다.")

RuntimeError: Unknown model (resnet-50)

In [78]:
import torch
import torch.nn as nn
import torch.optim as optim
import timm
from torchvision import transforms
from torch.utils.data import DataLoader
from tqdm import tqdm

# [수정 1] 과적합 방지를 위한 강력한 데이터 증강
train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomResizedCrop(224),       # 이미지를 랜덤하게 자름
    transforms.RandomHorizontalFlip(),       # 좌우 반전
    transforms.RandomRotation(15),           # 15도 내외 회전
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# [수정 2] 모델 구조 변경 (Dropout 추가)
model = timm.create_model('resnet50', pretrained=True, num_classes=4)

# classifier 대신 fc를 사용하세요!
model.fc = nn.Sequential(
    nn.Dropout(0.3),
    nn.Linear(model.fc.in_features, 512), # 여기도 model.fc.in_features로 유지
    nn.ReLU(),
    nn.Dropout(0.2),
    nn.Linear(512, 4)
)
model = model.to(DEVICE)

# [수정 3] Weight Decay 강화 (AdamW 사용)
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-1) 
criterion = nn.CrossEntropyLoss()

# 이후 학습 루프는 이전과 동일하게 사용하시면 됩니다.

In [75]:
import torch
import torch.nn as nn
import torch.optim as optim
import timm
from torchvision import transforms
from torch.utils.data import DataLoader
from tqdm import tqdm

# [수정 1] 과적합 방지를 위한 강력한 데이터 증강
train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomResizedCrop(224),       # 이미지를 랜덤하게 자름
    transforms.RandomHorizontalFlip(),       # 좌우 반전
    transforms.RandomRotation(15),           # 15도 내외 회전
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# [수정 2] 모델 구조 변경 (Dropout 추가)
model = timm.create_model('resnet50', pretrained=True, num_classes=4)

# classifier 대신 fc를 사용하세요!
model.fc = nn.Sequential(
    nn.Dropout(0.3),
    nn.Linear(model.fc.in_features, 1024),  # model.classifier 대신 model.fc.in_features
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(1024, 512),
    nn.ReLU(),
    nn.Linear(512, 4)
)
model = model.to(DEVICE)

# [수정 3] Weight Decay 강화 (AdamW 사용)
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-1) 
criterion = nn.CrossEntropyLoss()

# 이후 학습 루프는 이전과 동일하게 사용하시면 됩니다.

In [76]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import timm
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from tqdm import tqdm
from pathlib import Path

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
BATCH_SIZE = 32
EPOCHS = 3
MAX_TRAIN_SAMPLES = 500        # ⚡ 클래스당 500장 학습

print(f"🚀 [검증 없이 학습만 진행] 시스템 가동 (장치: {DEVICE})")

DATA_ROOT = Path(r"C:\Project\PyCharmMiscProject\PythonProject02\organized_bee_dataset")
TRAIN_IMAGE_DIR = DATA_ROOT / "train"

# ==========================================
# 데이터셋 클래스 (Train 모드만 사용)
# ==========================================
class FastBeeDataset(Dataset):
    def __init__(self, image_dir, transform=None, max_samples=500):
        self.image_paths = []
        self.labels = []
        self.transform = transform
        self.class_to_idx = {"알": 0, "백묵병": 1, "유충_응애": 2, "유충_정상": 3}
        
        print(f"🔍 [TRAIN] 폴더에서 선착순 {max_samples}장씩 자동 수집 중...")
        for class_name, idx in self.class_to_idx.items():
            class_folder = image_dir / class_name
            if class_folder.exists():
                all_images = [p for p in class_folder.glob("*") if p.suffix.lower() in ['.jpg', '.jpeg', '.png', '.bmp']]
                limited_images = all_images[:max_samples]
                for img_path in limited_images:
                    self.image_paths.append(img_path)
                    self.labels.append(idx)
                print(f"   - {class_name}: {len(limited_images)}장 확보완료")
        print(f"📦 [TRAIN] 총 {len(self.image_paths)}장 세팅 완료!\n")

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        try:
            img = Image.open(self.image_paths[idx]).convert('RGB')
            label = self.labels[idx]
            if self.transform:
                img = self.transform(img)
            return img, label
        except Exception:
            return self.__getitem__((idx + 1) % len(self.image_paths))

# ==========================================
# 모델 및 로더 설정
# ==========================================
model = timm.create_model('resnet50', pretrained=True, num_classes=4)
model = model.to(DEVICE)
transform = timm.data.create_transform(**timm.data.resolve_model_data_config(model))

train_dataset = FastBeeDataset(TRAIN_IMAGE_DIR, transform=transform, max_samples=MAX_TRAIN_SAMPLES)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)

optimizer = optim.AdamW(model.parameters(), lr=5e-4, weight_decay=1e-2)
criterion = nn.CrossEntropyLoss()

# ==========================================
# 학습 루프 (검증 과정 제거됨)
# ==========================================
for epoch in range(EPOCHS):
    model.train()
    train_loss, train_correct, train_total = 0.0, 0, 0
    
    train_pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]")
    for images, labels in train_pbar:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        train_total += labels.size(0)
        train_correct += (predicted == labels).sum().item()
        train_pbar.set_postfix(loss=f"{loss.item():.4f}", acc=f"{(train_correct/train_total)*100:.2f}%")
        
    print(f"📊 Epoch {epoch+1} | 최종 Train Acc: {(train_correct/train_total)*100:.2f}%\n")

# 최종 학습된 모델 저장
torch.save(model.state_dict(), 'microsoft_bee_model_no_val.pth')
print(f"💾 학습 완료! 'microsoft_bee_model_no_val.pth' 저장 성공!")

🚀 [검증 없이 학습만 진행] 시스템 가동 (장치: cuda)
🔍 [TRAIN] 폴더에서 선착순 500장씩 자동 수집 중...
   - 알: 500장 확보완료
   - 백묵병: 500장 확보완료
   - 유충_응애: 500장 확보완료
   - 유충_정상: 500장 확보완료
📦 [TRAIN] 총 2000장 세팅 완료!



Epoch 1/3 [Train]: 100%|███████████████████████████████████████████████████████████████████████████████| 63/63 [03:29<00:00,  3.32s/it, acc=90.15%, loss=0.2852]


📊 Epoch 1 | 최종 Train Acc: 90.15%



Epoch 2/3 [Train]: 100%|███████████████████████████████████████████████████████████████████████████████| 63/63 [03:37<00:00,  3.46s/it, acc=99.45%, loss=0.0030]


📊 Epoch 2 | 최종 Train Acc: 99.45%



Epoch 3/3 [Train]: 100%|███████████████████████████████████████████████████████████████████████████████| 63/63 [03:40<00:00,  3.51s/it, acc=99.95%, loss=0.0029]


📊 Epoch 3 | 최종 Train Acc: 99.95%

💾 학습 완료! 'microsoft_bee_model_no_val.pth' 저장 성공!


In [79]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import timm
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from tqdm import tqdm
from pathlib import Path

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
BATCH_SIZE = 32
EPOCHS = 3
MAX_TRAIN_SAMPLES = 500        # ⚡ 클래스당 500장 학습

print(f"🚀 [검증 없이 학습만 진행] 시스템 가동 (장치: {DEVICE})")

DATA_ROOT = Path(r"C:\Project\PyCharmMiscProject\PythonProject02\organized_bee_dataset")
TRAIN_IMAGE_DIR = DATA_ROOT / "train"

# ==========================================
# 데이터셋 클래스 (Train 모드만 사용)
# ==========================================
class FastBeeDataset(Dataset):
    def __init__(self, image_dir, transform=None, max_samples=500):
        self.image_paths = []
        self.labels = []
        self.transform = transform
        self.class_to_idx = {"알": 0, "백묵병": 1, "유충_응애": 2, "유충_정상": 3}
        
        print(f"🔍 [TRAIN] 폴더에서 선착순 {max_samples}장씩 자동 수집 중...")
        for class_name, idx in self.class_to_idx.items():
            class_folder = image_dir / class_name
            if class_folder.exists():
                all_images = [p for p in class_folder.glob("*") if p.suffix.lower() in ['.jpg', '.jpeg', '.png', '.bmp']]
                limited_images = all_images[:max_samples]
                for img_path in limited_images:
                    self.image_paths.append(img_path)
                    self.labels.append(idx)
                print(f"   - {class_name}: {len(limited_images)}장 확보완료")
        print(f"📦 [TRAIN] 총 {len(self.image_paths)}장 세팅 완료!\n")

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        try:
            img = Image.open(self.image_paths[idx]).convert('RGB')
            label = self.labels[idx]
            if self.transform:
                img = self.transform(img)
            return img, label
        except Exception:
            return self.__getitem__((idx + 1) % len(self.image_paths))

# ==========================================
# 모델 및 로더 설정
# ==========================================
model = timm.create_model('resnet50', pretrained=True, num_classes=4)
model = model.to(DEVICE)
transform = timm.data.create_transform(**timm.data.resolve_model_data_config(model))

train_dataset = FastBeeDataset(TRAIN_IMAGE_DIR, transform=transform, max_samples=MAX_TRAIN_SAMPLES)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)

optimizer = optim.AdamW(model.parameters(), lr=1e-5, weight_decay=5e-1)

criterion = nn.CrossEntropyLoss()

# ==========================================
# 학습 루프 (검증 과정 제거됨)
# ==========================================
for epoch in range(EPOCHS):
    model.train()
    train_loss, train_correct, train_total = 0.0, 0, 0
    
    train_pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]")
    for images, labels in train_pbar:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        train_total += labels.size(0)
        train_correct += (predicted == labels).sum().item()
        train_pbar.set_postfix(loss=f"{loss.item():.4f}", acc=f"{(train_correct/train_total)*100:.2f}%")
        
    print(f"📊 Epoch {epoch+1} | 최종 Train Acc: {(train_correct/train_total)*100:.2f}%\n")

# 최종 학습된 모델 저장
torch.save(model.state_dict(), 'microsoft_bee_model_no_val.pth')
print(f"💾 학습 완료! 'microsoft_bee_model_no_val.pth' 저장 성공!")

🚀 [검증 없이 학습만 진행] 시스템 가동 (장치: cuda)
🔍 [TRAIN] 폴더에서 선착순 500장씩 자동 수집 중...
   - 알: 500장 확보완료
   - 백묵병: 500장 확보완료
   - 유충_응애: 500장 확보완료
   - 유충_정상: 500장 확보완료
📦 [TRAIN] 총 2000장 세팅 완료!



Epoch 1/3 [Train]: 100%|███████████████████████████████████████████████████████████████████████████████| 63/63 [03:43<00:00,  3.55s/it, acc=52.00%, loss=1.3042]


📊 Epoch 1 | 최종 Train Acc: 52.00%



Epoch 2/3 [Train]: 100%|███████████████████████████████████████████████████████████████████████████████| 63/63 [03:37<00:00,  3.44s/it, acc=75.95%, loss=1.3182]


📊 Epoch 2 | 최종 Train Acc: 75.95%



Epoch 3/3 [Train]: 100%|███████████████████████████████████████████████████████████████████████████████| 63/63 [02:51<00:00,  2.72s/it, acc=83.50%, loss=1.3059]


📊 Epoch 3 | 최종 Train Acc: 83.50%

💾 학습 완료! 'microsoft_bee_model_no_val.pth' 저장 성공!


# 학습 결과 ==========================================

In [50]:
import torch

# 파이썬 pickle 대신 PyTorch의 torch.load를 사용합니다.
# 파일명이 'optimized_bee_model_backup.pth'이므로 경로를 올바르게 맞춰줍니다.
model_data = torch.load('microsoft_bee_model_no_val.pth', map_location='cpu')

print(model_data.keys())  # 내부 레이어 이름 확인

odict_keys(['conv1.weight', 'bn1.weight', 'bn1.bias', 'bn1.running_mean', 'bn1.running_var', 'bn1.num_batches_tracked', 'layer1.0.conv1.weight', 'layer1.0.bn1.weight', 'layer1.0.bn1.bias', 'layer1.0.bn1.running_mean', 'layer1.0.bn1.running_var', 'layer1.0.bn1.num_batches_tracked', 'layer1.0.conv2.weight', 'layer1.0.bn2.weight', 'layer1.0.bn2.bias', 'layer1.0.bn2.running_mean', 'layer1.0.bn2.running_var', 'layer1.0.bn2.num_batches_tracked', 'layer1.0.conv3.weight', 'layer1.0.bn3.weight', 'layer1.0.bn3.bias', 'layer1.0.bn3.running_mean', 'layer1.0.bn3.running_var', 'layer1.0.bn3.num_batches_tracked', 'layer1.0.downsample.0.weight', 'layer1.0.downsample.1.weight', 'layer1.0.downsample.1.bias', 'layer1.0.downsample.1.running_mean', 'layer1.0.downsample.1.running_var', 'layer1.0.downsample.1.num_batches_tracked', 'layer1.1.conv1.weight', 'layer1.1.bn1.weight', 'layer1.1.bn1.bias', 'layer1.1.bn1.running_mean', 'layer1.1.bn1.running_var', 'layer1.1.bn1.num_batches_tracked', 'layer1.1.conv2.we

In [61]:
import torch
import timm

# 1. 모델 아키텍처 정의
# 저장된 파일의 구조와 정확히 일치하도록 num_classes를 설정해야 합니다.
# (파일이 4개 클래스용이므로 4로 지정)
model = timm.create_model('resnet50', pretrained=False, num_classes=4)

# 2. 가중치 파일 경로 설정
# 사용하고 싶은 파일명을 입력하세요 ('optimized_bee_model.pth' 또는 '..._backup.pth')
model_path = 'microsoft_bee_model_no_val.pth' 

# 3. 가중치 로드
try:
    # map_location='cpu'를 사용하여 CPU 환경에서도 안전하게 불러옵니다.
    model_data = torch.load(model_path, map_location='cpu')
    
    # 모델에 가중치 입히기
    model.load_state_dict(model_data)
    print(f"성공: '{model_path}'의 가중치가 모델에 정상적으로 로드되었습니다.")
    
except Exception as e:
    print(f"오류 발생: 가중치 로드 실패 - {e}")

# 4. 모델 모드 설정
# 추론(테스트)을 위해 모델을 eval 모드로 전환합니다.
model.eval()

# 5. 확인 출력 (선택 사항)
# 모델이 정상적으로 로드되었는지 최종 확인
print("모델이 추론 준비 완료 상태입니다.")

# 이후 이 model 객체를 사용하여 실제 이미지를 예측(predict)할 수 있습니다.

성공: 'microsoft_bee_model_no_val.pth'의 가중치가 모델에 정상적으로 로드되었습니다.
모델이 추론 준비 완료 상태입니다.


In [62]:
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"사용 중인 장치: {device}")

사용 중인 장치: cuda


In [65]:
import cv2
import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image
from torchvision import transforms

# 1. 테스트용 고화질 이미지에 최적화된 전처리 (학습 시와 동일한 구조)
test_transform = transforms.Compose([
    transforms.Resize((256, 256)),      # 학습 때보다 약간 큰 사이즈로 리사이징
    transforms.CenterCrop(224),         # 중앙을 잘라내어 노이즈 제거
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

def predict_image_robust(image_path, model):
    # 1. 모델을 현재 장치(device)로 이동 (이미 되어있다면 중복되어도 안전합니다)
    model = model.to(device)
    model.eval()
    
    # 2. 이미지 로드 및 전처리
    img = Image.open(image_path).convert('RGB')
    img_tensor = test_transform(img).unsqueeze(0).to(device) # 데이터를 GPU로
    
    # 3. 추론
    with torch.no_grad():
        output = model(img_tensor)
        probabilities = F.softmax(output[0], dim=0)
        prediction = torch.argmax(probabilities, dim=0).item()
        confidence = probabilities[prediction].item() * 100
        
    return prediction, confidence

# 실행 예시
labels = ["알", "백묵병", "응애", "정상"]
pred_idx, conf = predict_image_robust('test.jpg', model)

print(f"분석 결과: {labels[pred_idx]} (확신도: {conf:.2f}%)")

분석 결과: 응애 (확신도: 99.99%)


In [28]:
# from PIL import Image
# from timm.data import create_transform
# from timm.data.constants import IMAGENET_DEFAULT_MEAN, IMAGENET_DEFAULT_STD

# # 전처리 함수 정의 (이미지 크기 조절 및 정규화)
# transform = create_transform(
#     input_size=224, # efficientnet_b0의 기본 입력 사이즈
#     mean=IMAGENET_DEFAULT_MEAN,
#     std=IMAGENET_DEFAULT_STD
# )

# def predict_image(image_path, model):
#     img = Image.open(image_path).convert('RGB')
#     img_tensor = transform(img).unsqueeze(0) # 배치 차원 추가 (1, 3, 224, 224)
    
#     with torch.no_grad(): # 역전파 비활성화 (메모리 절약)
#         output = model(img_tensor)
#         prediction = torch.argmax(output, dim=1)
        
#     return prediction.item()

# # 사용 예시
# image_path = 'test.jpg' # 실제 이미지 파일 경로
# result = predict_image(image_path, model)
# print(f"예측된 클래스 인덱스: {result}")

In [67]:
# 테스트할 이미지 파일 경로들을 리스트로 만듭니다.
# 예: 폴더별로 한 장씩 골라보세요.
test_images = ['test.jpg', 'test1.jpg', 'test2.jpg']

for img_path in test_images:
    # 수정 전
# idx = predict_image(img_path, model)

# 수정 후: 43번 셀에서 만든, GPU 장치 대응이 완료된 함수를 사용하세요.
    pred_idx, conf = predict_image_robust(img_path, model)
    print(f"이미지 {img_path} -> 예측된 클래스: {labels[pred_idx]} (확신도: {conf:.2f}%)")

이미지 test.jpg -> 예측된 클래스: 응애 (확신도: 99.99%)
이미지 test1.jpg -> 예측된 클래스: 정상 (확신도: 99.04%)
이미지 test2.jpg -> 예측된 클래스: 백묵병 (확신도: 69.04%)


In [69]:
import cv2
import numpy as np
import torch
import torch.nn.functional as F
from PIL import ImageFont, ImageDraw, Image

# 1. 이미지 로드 (이 셀 내에서 한꺼번에 처리)
image_path = 'test2.jpg' 
img_cv = cv2.imread(image_path)
img_rgb = cv2.cvtColor(img_cv, cv2.COLOR_BGR2RGB)
img_pil = Image.fromarray(img_rgb)

# 2. 전처리 (이전 셀에서 정의한 transform 사용)
img_tensor = transform(img_pil).unsqueeze(0).to(device)

# 3. 모델 추론
model.eval()
with torch.no_grad():
    output = model(img_tensor)
    probabilities = F.softmax(output[0], dim=0)
    prediction = torch.argmax(output, dim=1).item()
    confidence = probabilities[prediction].item() * 100

# --- 추가된 출력 부분 ---
labels = ["알", "백묵병", "응애", "정상"]
print("-" * 30)
print(f"분석 결과: {labels[prediction]}")
print(f"확신도(Confidence): {confidence:.2f}%")
print("-" * 30)
# ----------------------

# 4. 한글 및 퍼센트 출력 준비 (이미지용)
fontpath = "malgun.ttf" 
font = ImageFont.truetype(fontpath, 40)
draw = ImageDraw.Draw(img_pil)
text = f"{labels[prediction]} ({confidence:.1f}%)"
draw.text((50, 50), text, font=font, fill=(0, 255, 0))

# 5. OpenCV로 결과 출력
img_result = cv2.cvtColor(np.array(img_pil), cv2.COLOR_RGB2BGR)
cv2.imshow('Bee Detection', img_result)
cv2.waitKey(0)
cv2.destroyAllWindows()

------------------------------
분석 결과: 백묵병
확신도(Confidence): 99.98%
------------------------------


In [58]:
import cv2
import numpy as np
import torch
import torch.nn.functional as F
from PIL import ImageFont, ImageDraw, Image

# 1. 이미지 로드 (이 셀 내에서 한꺼번에 처리)
image_path = 'test.jpg' 
img_cv = cv2.imread(image_path)
img_rgb = cv2.cvtColor(img_cv, cv2.COLOR_BGR2RGB)
img_pil = Image.fromarray(img_rgb)

# 2. 전처리 (이전 셀에서 정의한 transform 사용)
img_tensor = transform(img_pil).unsqueeze(0).to(device)

# 3. 모델 추론
model.eval()
with torch.no_grad():
    output = model(img_tensor)
    probabilities = F.softmax(output[0], dim=0)
    prediction = torch.argmax(output, dim=1).item()
    confidence = probabilities[prediction].item() * 100

# --- 추가된 출력 부분 ---
labels = ["알", "백묵병", "응애", "정상"]
print("-" * 30)
print(f"분석 결과: {labels[prediction]}")
print(f"확신도(Confidence): {confidence:.2f}%")
print("-" * 30)
# ----------------------

# 4. 한글 및 퍼센트 출력 준비 (이미지용)
fontpath = "malgun.ttf" 
font = ImageFont.truetype(fontpath, 40)
draw = ImageDraw.Draw(img_pil)
text = f"{labels[prediction]} ({confidence:.1f}%)"
draw.text((50, 50), text, font=font, fill=(0, 255, 0))

# 5. OpenCV로 결과 출력
img_result = cv2.cvtColor(np.array(img_pil), cv2.COLOR_RGB2BGR)
cv2.imshow('Bee Detection', img_result)
cv2.waitKey(0)
cv2.destroyAllWindows()

------------------------------
분석 결과: 응애
확신도(Confidence): 99.99%
------------------------------


In [30]:
# 1. 학습 데이터 수 확인
num_train = len(train_dataset.image_paths)
print(f"학습 데이터 총 개수: {num_train}장")

# 클래스별 개수 확인
for class_name, idx in train_dataset.class_to_idx.items():
    count = sum(1 for label in train_dataset.labels if label == idx)
    print(f" - {class_name}: {count}장")

print("-" * 30)

# 2. 검증 데이터 수 확인
num_val = len(val_dataset.image_paths)
print(f"검증 데이터 총 개수: {num_val}장")

# 클래스별 개수 확인
for class_name, idx in val_dataset.class_to_idx.items():
    count = sum(1 for label in val_dataset.labels if label == idx)
    print(f" - {class_name}: {count}장")

학습 데이터 총 개수: 2000장
 - 알: 500장
 - 백묵병: 500장
 - 유충_응애: 500장
 - 유충_정상: 500장
------------------------------
검증 데이터 총 개수: 200장
 - 알: 50장
 - 백묵병: 50장
 - 유충_응애: 50장
 - 유충_정상: 50장


In [38]:
from pathlib import Path

def count_all_files_in_directory(root_dir):
    # 'train' 폴더와 'val' 폴더 각각에 대해 확인
    for folder_name in ['train', 'labels']:
        target_dir = Path(root_dir) / folder_name
        print(f"\n [{folder_name.upper()} 전체 경로 탐색 중...]")
        
        if not target_dir.exists():
            print(f"경로를 찾을 수 없습니다: {target_dir}")
            continue
            
        total_in_folder = 0
        # 각 클래스 폴더별로 파일 개수 세기
        for class_folder in target_dir.iterdir():
            if class_folder.is_dir():
                # 이미지 파일만 카운트 (jpg, jpeg, png, bmp)
                files = [p for p in class_folder.glob("*") if p.suffix.lower() in ['.jpg', '.jpeg', '.png', '.bmp','.json']]
                count = len(files)
                print(f" - {class_folder.name}: {count}장")
                total_in_folder += count
        
        print(f"총 파일 수: {total_in_folder}장")

# DATA_ROOT는 현재 사용 중인 경로로 설정되어 있습니다.
# 만약 경로가 다르다면 r"전체_경로_입력"으로 직접 수정해주세요.
DATA_ROOT = Path(r"C:\Project\PyCharmMiscProject\PythonProject02\organized_bee_dataset")
count_all_files_in_directory(DATA_ROOT)


 [TRAIN 전체 경로 탐색 중...]
 - 백묵병: 9450장
 - 알: 20035장
 - 유충_응애: 82778장
 - 유충_정상: 28196장
총 파일 수: 140459장

 [LABELS 전체 경로 탐색 중...]
 - 백묵병: 9450장
 - 알: 20035장
 - 유충_응애: 92636장
 - 유충_정상: 28196장
총 파일 수: 150317장
